# Backdoor Attack, Activation Clustering, and Repair on CIFAR-10 with ResNet-18

In [ ]:
import os
import json
import math
import copy
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA, PCA
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as T
import torchvision.models as models

# -----------------------------
# Config
# -----------------------------
CONFIG = {
    "seed": 42,
    "data_dir": "./data",
    "out_dir": "./outputs",
    "num_classes": 10,
    "target_class": 0,              # recommended by assignment
    "val_per_class": 500,           # balanced clean validation split
    "poison_rate": 0.05,            # 5% of non-target training samples
    "trigger_size": 3,              # 3x3 white square
    "trigger_value": 1.0,           # pixel value in [0,1]
    "trigger_offset": 1,            # 1 pixel away from bottom-right border
    "batch_size": 128,
    "num_workers": 2,
    "epochs": 30,
    "lr": 0.1,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "ica_components": 10,
    "kmeans_n_init": 20,
    "feature_batch_size": 256,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

OUT_DIR = Path(CONFIG["out_dir"])
FIG_DIR = OUT_DIR / "figures"
CKPT_DIR = OUT_DIR / "checkpoints"
META_DIR = OUT_DIR / "metadata"

for d in [OUT_DIR, FIG_DIR, CKPT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Device:", CONFIG["device"])
print("Outputs will be saved to:", OUT_DIR.resolve())

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism settings
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

### Dataset split and poisoning design
We create:
- a balanced clean validation split of 500 samples per class,
- a clean training split from the remaining CIFAR-10 training samples,
- a poisoned training split by poisoning 5% of non-target training examples.

Poisoned samples:
- receive a fixed 3x3 white square trigger in the bottom-right corner,
- are relabeled to the target class.

In [ ]:
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
])

eval_transform = T.Compose([
    T.ToTensor(),
])

In [ ]:
base_train_raw = torchvision.datasets.CIFAR10(
    root=CONFIG["data_dir"], train=True, download=True
)
base_test_raw = torchvision.datasets.CIFAR10(
    root=CONFIG["data_dir"], train=False, download=True
)

train_targets = np.array(base_train_raw.targets)

def make_balanced_val_split(targets, val_per_class=500, seed=42, num_classes=10):
    rng = np.random.default_rng(seed)
    val_indices = []
    train_indices = []

    for cls in range(num_classes):
        cls_indices = np.where(targets == cls)[0]
        rng.shuffle(cls_indices)
        val_cls = cls_indices[:val_per_class]
        train_cls = cls_indices[val_per_class:]
        val_indices.extend(val_cls.tolist())
        train_indices.extend(train_cls.tolist())

    return np.array(train_indices), np.array(val_indices)

train_indices, val_indices = make_balanced_val_split(
    train_targets,
    val_per_class=CONFIG["val_per_class"],
    seed=CONFIG["seed"],
    num_classes=CONFIG["num_classes"],
)

print("Train split:", len(train_indices))
print("Val split:", len(val_indices))

pd.DataFrame({"train_indices": train_indices}).to_csv(META_DIR / "train_indices.csv", index=False)
pd.DataFrame({"val_indices": val_indices}).to_csv(META_DIR / "val_indices.csv", index=False)

#### Trigger and poisoning utilities

In [ ]:
def add_trigger_to_tensor(img_tensor, trigger_size=3, value=1.0, offset=1):
    """
    img_tensor: torch.Tensor of shape [C, H, W] with values in [0,1]
    Places a square trigger in the bottom-right corner.
    """
    x = img_tensor.clone()
    _, h, w = x.shape

    r_end = h - offset
    c_end = w - offset
    r_start = r_end - trigger_size
    c_start = c_end - trigger_size

    x[:, r_start:r_end, c_start:c_end] = value
    return x

def visualize_trigger_example(dataset, idx=0):
    img, label = dataset[idx]
    triggered = add_trigger_to_tensor(
        img,
        trigger_size=CONFIG["trigger_size"],
        value=CONFIG["trigger_value"],
        offset=CONFIG["trigger_offset"]
    )

    fig, axes = plt.subplots(1, 2, figsize=(5, 3))
    axes[0].imshow(np.transpose(img.numpy(), (1, 2, 0)))
    axes[0].set_title(f"Original (label={label})")
    axes[0].axis("off")

    axes[1].imshow(np.transpose(triggered.numpy(), (1, 2, 0)))
    axes[1].set_title("Triggered")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

temp_eval_train = torchvision.datasets.CIFAR10(
    root=CONFIG["data_dir"], train=True, download=False, transform=eval_transform
)
visualize_trigger_example(temp_eval_train, idx=0)